# Semana 07
## Calculos analiticos, share of total, running totals y análogos LOD

**Objetivo**: construir métricas derivadas y entender cómo cambia el resultado según el nivel de cálculo.

**Herramientas teoricas de la semana**
- metricas derivadas
- share of total
- running total
- analogos de `FIXED` con `groupby.transform`


### Agenda sugerida de 4 horas
- 0:00 - 0:30: teoria del nivel de calculo
- 0:30 - 1:20: metricas derivadas y comparacion de niveles
- 1:20 - 2:10: percent of total y running totals
- 2:10 - 3:20: benchmark tipo `FIXED`
- 3:20 - 4:00: validacion de formulas y exportables


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from _shared import (
    make_base_sales,
    introduce_quality_issues,
    profile_dataframe,
    clean_sales_data,
    build_star_schema,
    save_for_tableau,
    ensure_output_dir,
    contrast_ratio,
    make_high_dimensional_dataset,
)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
WEEK = "week-07"
OUTPUT_DIR = ensure_output_dir(WEEK)

clean, _ = clean_sales_data(introduce_quality_issues(make_base_sales(n=2200, seed=17), seed=17))
clean['ticket_avg_proxy'] = clean['sales'] / clean['quantity']
clean['region_total_sales'] = clean.groupby('region')['sales'].transform('sum')  # analogo FIXED [Region]
clean['share_of_region_sales'] = clean['sales'] / clean['region_total_sales']
clean.head()


In [ ]:
monthly = (
    clean.assign(month=clean['order_date'].dt.to_period('M').dt.to_timestamp())
    .groupby('month', as_index=False)
    .agg(monthly_sales=('sales', 'sum'))
    .sort_values('month')
)
monthly['running_total'] = monthly['monthly_sales'].cumsum()
monthly['rolling_3m'] = monthly['monthly_sales'].rolling(3).mean()
monthly.head()


In [ ]:
category_view = (
    clean.groupby(['region', 'category'], as_index=False)
    .agg(total_sales=('sales', 'sum'))
)
category_view['pct_of_region'] = category_view['total_sales'] / category_view.groupby('region')['total_sales'].transform('sum')
category_view.head()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
monthly.plot(x='month', y=['monthly_sales', 'running_total'], ax=axes[0], title='Venta mensual y running total', color=['#6b7280', '#111827'])
sns.barplot(data=category_view, x='region', y='pct_of_region', hue='category', palette='Greys', ax=axes[1])
axes[1].set_title('Participacion por categoria dentro de cada region')
plt.tight_layout()


In [ ]:
metric_definitions = pd.DataFrame([
    {'metric': 'ticket_avg_proxy', 'definition': 'sales / quantity', 'level': 'row'},
    {'metric': 'region_total_sales', 'definition': 'sum(sales) por region', 'level': 'fixed-like region'},
    {'metric': 'share_of_region_sales', 'definition': 'sales / region_total_sales', 'level': 'row vs region benchmark'},
    {'metric': 'running_total', 'definition': 'cumulative sum monthly_sales', 'level': 'table over time'},
])
metric_definitions


In [ ]:
save_for_tableau(monthly, WEEK, 'monthly_metrics')
save_for_tableau(category_view, WEEK, 'category_region_shares')
save_for_tableau(metric_definitions, WEEK, 'metric_definitions')


### Uso teorico de herramientas
- `groupby.transform` funciona como análogo pedagógico de un `FIXED`.
- `cumsum` y `rolling` conectan con table calculations.
- Documentar definiciones de métricas es indispensable para evitar ambigüedad en Tableau.
